## Causal Language Modeling (CLM) - Preprocess, Training and Inference

In this lab, we will explore **Causal Language Modeling (CLM)**, which is a core task in training autoregressive language models like GPT-2. CLM is the process of predicting the next word in a sequence, given the previous words. This type of modeling forms the backbone of text generation tasks, where the model learns to generate coherent text by focusing only on previous tokens in the sequence.

The lab is divided into three major sections:
1. **Preprocessing**: Preparing a dataset and the labels for the CLM task and tokenizing them.
2. **Training**: Fine-tuning a pre-trained language model like GPT-2 on a specific dataset using the CLM task.
3. **Inference**: Evaluating the model’s performance by generating text based on input prompts.

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import load_dataset

### 1. Preprocessing

1. **Load the Domain-Specific Dataset**:
   - The first step in fine-tuning GPT-2 is to load a dataset that is specific to the domain of interest. In this case, we are using a publicly available **medical dataset** from the **PubMed** collection. PubMed contains a vast number of medical articles, and fine-tuning on such a dataset can help GPT-2 generate more accurate and context-specific medical text.
   - The dataset we're using is `"japhba/pubmed_simple"`, which is a simplified version of PubMed data. This dataset can be easily accessed using the `datasets` library from Hugging Face.


In [2]:
ds = load_dataset("japhba/pubmed_simple", split="train")
train_dataset = ds.shuffle(seed=42).select(range(1000)) # 1000 samples for train
eval_dataset = ds.shuffle(seed=42).select(range(1000, 1500)) # 500 samples for test

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


We can check the contents of any one of the examples in the dataset to understand the structure of the data.

We see that the each entry is a dictionary with two keys:
- "abstract": The abstract of the medical article
- "country": The country where the article was published (we will not use this information in this lab)

2. **Tokenize the Dataset**:
   - The next step is to **tokenize** the data so that it can be processed by GPT-2.
   - GPT-2 does not natively use a padding token, since it does not require fixed length inputs. For this reason, we will substite it with the EOS token as suggested by transformers library (remember, we are also passing an attention mask, so whatever value is used for padding will be ignored by the model!)

In [3]:
# TODO: Tokenize the Dataset using the GPT-2 tokenizer
# Hint: Use the `map` method of the dataset object
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Set padding token to EOS token

def tokenize_function(samples):
    # Tokenize the text column `abstract` in the dataset (set `truncation=True` and `padding="max_length"`)
    # NOTE: You can either use tokenzier as a global variable, or
    # use a closure to capture the tokenizer variable

    return tokenizer(samples['abstract'],truncation = True, padding = "max_length")

# NOTE: you should map the dataset using the tokenize function.
# (You can optionally consider using the argument `batched=True` for better performance,
# as long as tokenize_function can handle the batch! [note2: the tokenizer can do that!])
tokenized_dataset = train_dataset.map(tokenize_function , batched=True)
tokenized_dataset_eval = eval_dataset.map(tokenize_function , batched=True)

# The .map() function is a highly optimized tool built into Hugging Face datasets.
# Instead of writing a slow Python for loop to process your 1,000 rows one by one, map applies your custom function to the whole dataset.
# batched=True: This is the secret to high performance. It grabs chunks of data (usually 1,000 rows at a time)
# and tokenizes them simultaneously in C/Rust under the hood. It is drastically faster than tokenizing line by line.


# You take your raw text dataset, define strict rules so that every text string is converted into a uniform-length array of numbers,
# and then efficiently apply those rules to both your training and testing sets.


3. **Add labels to the Dataset for Next Token Prediction**:
   - In this step, we will add the **labels** that will be used during the next token prediction task.
   - In autoregressive language modeling, the **labels** represent the same sequence as the input, shifted one token to the right. This is because the model is trained to predict the next token in the sequence given the previous tokens.
   - The shifting of the tokens is already handled automatically by the `Trainer` class. We just pass an extra attribute named `labels` to the dataset (when this argument is passed to the model, it will know to compute the loss for us!)

In [4]:
# TODO: Add labels to the dataset
def add_labels(samples):
    # NOTE: The labels, in causal modeling, should be the same as the input_ids
    samples["labels"] = samples["input_ids"]
    return samples

tokenized_dataset = tokenized_dataset.map(add_labels, batched= True)
tokenized_dataset_eval = tokenized_dataset_eval.map(add_labels, batched= True)

In [5]:
print(ds)
print(train_dataset[0])

print(tokenized_dataset)
print(tokenized_dataset[0]['labels'])

Dataset({
    features: ['abstract', 'country'],
    num_rows: 3116832
})
{'abstract': 'Purpose The identification of abnormalities that are relatively rare within otherwise normal anatomy is a major challenge for deep learning in the semantic segmentation of medical images. The small number of samples of the minority classes in the training data makes the learning of optimal classification challenging, while the more frequently occurring samples of the majority class hamper the generalization of the classification boundary between infrequently occurring target objects and classes. In this paper, we developed a novel generative multi-adversarial network, called Ensemble-GAN, for mitigating this class imbalance problem in the semantic segmentation of abdominal images.Method The Ensemble-GAN framework is composed of a single-generator and a multi-discriminator variant for handling the class imbalance problem to provide a better generalization than existing approaches. The ensemble model 

### 2. Training

1. **Fine-Tune the GPT-2 Model**:
   - Set up the model and finetune it using the medical dataset.
   - The pipeline to be followed is the same that we have already seen in the previous lab (`lab03 - 01-bert`)

In [ ]:
# Set Training Parameters
training_args = TrainingArguments(
    output_dir="./gpt2-medical-finetuned",
    # overwrite_output_dir=True,
    num_train_epochs=1, # model will look at your 1,000 medical abstracts exactly one time usually higher
    per_device_train_batch_size=6,
    save_total_limit=2,
    # As the model trains, it saves backup copies of itself (checkpoints).
    # This tells your computer to only keep the 2 most recent backups and delete older ones so you don't fill up your entire hard drive.
    logging_dir="./logs", # where to save the training statistics
    logging_steps=100,
    # The trainer will print an update to your screen (showing the current error rate, or "loss") every 100 batches.
    eval_steps=10,
    # Every 10 steps, the model will pause training, look at your eval_dataset (the 500 test abstracts), and check to see if it can accurately predict them.
    eval_strategy="steps",
    # Evaluate the model based on a certain number of steps, rather than waiting for the whole epoch to finish.
)

# TODO: Initialize GPT-2 Model
model = AutoModelForCausalLM.from_pretrained(model_name)

# TODO: Fine-Tune the Model with the `Trainer` method and pass also the eval_dataset
# Writing a PyTorch training loop from scratch requires hundreds of lines of complex calculus and gradient-descent code.
# Hugging Face built the Trainer class to do all that for you.
trainer = Trainer(
    model=model, # brain
    args=training_args, # rules
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset_eval,
)

trainer.train()

# Save the Fine-Tuned Model
model.save_pretrained("./gpt2-medical-finetuned")


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


### 3. Inference

1. **Compare Text Generation Before and After Fine-Tuning**:
   - Generate text using both the original pre-trained GPT-2 model and the fine-tuned model.
   - Provide the same input prompt and observe the differences in the outputs.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# TODO: Load the pre-trained and fine-tuned GPT-2 models
pretrained_model = AutoModelForCausalLM.from_pretrained("gpt2")
finetuned_model = AutoModelForCausalLM.from_pretrained("./gpt2-medical-finetuned")

# TODO: Tokenize the prompt
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

prompt = "The patient presents with chest pain and shortness of breath."

inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs['input_ids']

In [ ]:
# TODO: Generate Output of the Model Before Fine-Tuning (use `generate` and then `decode` methods of the model and tokenizer)
output_pretrained = pretrained_model.generate(input_ids,do_sample=True, max_length=100, temperature=1.0, pad_token_id=tokenizer.eos_token_id)
# (assigning pad_token_id avoids a warning)
generated_pretrained = finetuned

print("**Before Fine-Tuning (Pre-Trained GPT-2)**:")
print(generated_pretrained)

In [ ]:
# TODO: Generate Output of the Model After Fine-Tuning
output_finetuned = ...
generated_finetuned = ...

print("**After Fine-Tuning (Fine-Tuned on Medical Dataset)**:")
print(generated_finetuned)

<span style="color:red">Extra stuff!</span>

Training the model in this way produces batches with potentially very different lengths. This can be inefficient, as the model will have to pad the sequences to the length of the longest sequence in the batch.

To avoid this, we can use a technique called **Dynamic Padding**. This technique groups the sequences in the batch by length and pads them to the length of the longest sequence in each group. This way, the model only has to pad the sequences to the length of the longest sequence in each group, which can significantly reduce the amount of padding required.

As a first exercise, quantify the number of pad tokens being used in various situations:
1. You pad all batches to the maximum allowed sequence length (1024 for GPT-2, this is what we used so far)
2. You pad the entire batch to the length of the longest sequence in the batch (generate the batches by randomly sampling sentences)
3. You pad the entire batch to the length of the longest sequence in the batch (generate the batches by placing sentences of similar lengths together)

Next, introduce dynamic padding and compare the execution times of the previous execution and the one with dynamic padding.

You can use the following resources to help you with this exercise:
- `group_by_length` parameter ([TrainingArguments](https://huggingface.co/docs/transformers/v4.46.0/en/main_classes/trainer#transformers.TrainingArguments.group_by_length)) (parameter to group together samples with similar lengths)
- `DataCollatorForSeq2Seq` ([DataCollatorForSeq2Seq](https://huggingface.co/docs/transformers/main_classes/data_collator#transformers.DataCollatorForSeq2Seq)) (collator function that aggregates samples into batches and pads them to the maximum length of the batch)

Note: you may find that the validation losses you observe may be different from the previous ones. This is because the cross entropy loss is computed as an average across tokens, and the number of tokens in a batch can vary depending on the padding strategy used.